In [1]:
import random

n_q = 4    # number of qubits
m = 2**n_q    # vector dimension
n_shots = None    # measurement

In [2]:
import numpy as np
def coeff_mat_1d_Dirichlet(M):
    A = 2*np.identity(M-1)
    for i in range(M-2):
        A[i][i+1] = -1
        A[i+1][i] = -1
    return A

mat_D = coeff_mat_1d_Dirichlet(m+1)
rhs = np.array([1 for i in range(2**n_q)])
normalized_rhs = rhs/np.linalg.norm(rhs)

sol = np.linalg.solve(mat_D, rhs)
normalized_sol = sol/np.linalg.norm(sol)

print(normalized_sol)

[0.07354631 0.13789934 0.19305907 0.23902552 0.27579867 0.30337854
 0.32176512 0.33095841 0.33095841 0.32176512 0.30337854 0.27579867
 0.23902552 0.19305907 0.13789934 0.07354631]


In [3]:
A_oper_list = []
A_oper_list.append('0'*n_q)
for i in range(n_q):
    base_0 = '0'*(n_q-i-1)
    oper_simple = '1'*(i+1)
    A_oper_list.append(base_0+oper_simple)

In [4]:
import pennylane as pl
dev_get_state = pl.device("default.qubit", wires=n_q)

@pl.qnode(dev_get_state)
def get_ansatz_state(theta):
    ## ansatz unitary
    p = int((len(theta)-n_q)/(2*(n_q-1)))  # depth of ansatz

    for i in range(n_q):
        pl.RY(theta[i], wires=i)
    # pl.Barrier(wires=[i for i in range(n_q)])
    for iter in range(p):
        for i in range(n_q//2):
            pl.CZ(wires=[2*i, 2*i+1])
            pl.RY(theta[n_q+2*iter*(n_q-1)+2*i], wires=2*i)
            pl.RY(theta[n_q+2*iter*(n_q-1)+2*i+1], wires=2*i+1)
        if n_q != 2:
            for i in range((n_q-1)//2):
                pl.CZ(wires=[2*i+1, 2*i+2])
                pl.RY(theta[n_q+2*iter*(n_q-1)+2*(n_q//2)+2*i], wires=2*i+1)
                pl.RY(theta[n_q+2*iter*(n_q-1)+2*(n_q//2)+2*i+1], wires=2*i+2)
        # pl.Barrier(wires=[i for i in range(n_q)])
    return pl.state()

def prob_to_float(results, bitstring):
    return results[int(bitstring, 2)]

In [5]:
dev_denom = pl.device("default.qubit", wires=n_q, shots=n_shots)
@pl.qnode(dev_denom)
def qc_denominator_oper(A_oper, theta):
    ## ansatz unitary
    p = int((len(theta)-n_q)/(2*(n_q-1)))  # depth of ansatz
    for i in range(n_q):
        pl.RY(theta[i], wires=i)
    for iter in range(p):
        for i in range(n_q//2):
            pl.CZ(wires=[2*i, 2*i+1])
            pl.RY(theta[n_q+2*iter*(n_q-1)+2*i], wires=2*i)
            pl.RY(theta[n_q+2*iter*(n_q-1)+2*i+1], wires=2*i+1)
        if n_q != 2:
            for i in range((n_q-1)//2):
                pl.CZ(wires=[2*i+1, 2*i+2])
                pl.RY(theta[n_q+2*iter*(n_q-1)+2*(n_q//2)+2*i], wires=2*i+1)
                pl.RY(theta[n_q+2*iter*(n_q-1)+2*(n_q//2)+2*i+1], wires=2*i+2)
    # pl.Barrier(wires=[i for i in range(n_q)])
    ## measurement
    if A_oper == '0'*n_q:
        return pl.probs(wires=[i for i in range(n_q)])
    else:
        n_cr = len(A_oper) - A_oper.count('0')
        for i in range(1, n_cr):
            pl.CNOT(wires=[n_q-i-1, n_q-i])
        pl.Hadamard(wires=n_q-n_cr)
        measure_list = [n_q-n_cr+i for i in range(n_cr)]
        return pl.probs(wires=measure_list)
    
def expectation_denominator_oper(A_oper, results):
    if A_oper == '0' * n_q:
        expectation = 2
    elif A_oper.count('1') == 1:
        expectation = prob_to_float(results, '0') - prob_to_float(results, '1')
        expectation = -expectation
    else:
        expectation = prob_to_float(results, '01'+'0'*(A_oper.count('1')-2)) - prob_to_float(results, '11'+'0'*(A_oper.count('1')-2))
        expectation = -expectation
    return expectation

def cost_denominator(theta):
    def cost_denominator_oper(A_oper):
        counts = qc_denominator_oper(A_oper, theta)
        cost = expectation_denominator_oper(A_oper, results=counts)
        return cost
    denom_cost = 0
    for A_oper in A_oper_list:
        denom_cost += cost_denominator_oper(A_oper)
    return denom_cost

In [6]:
### example
p = 2
num_param = n_q + 2*p*(n_q-1)
list_param = np.array([i for i in range(16)])*np.pi/8
test_theta = np.array([random.choice(list_param) for i in range(num_param)])
state = np.array(get_ansatz_state(test_theta))
test_mat = mat_D
exact_result = np.dot(state, test_mat @ state.transpose())
print('exact_result:', exact_result)
qc_result = 0
for item in A_oper_list:
    results = qc_denominator_oper(A_oper=item, theta=test_theta)
    result = expectation_denominator_oper(A_oper=item, results=results)
    print('A_oper:', item, 'result:', result)
    qc_result += result
print('qc_result:', qc_result)

exact_result: (3.424287082602717+0j)
A_oper: 0000 result: 2
A_oper: 0001 result: 0.709936836763559
A_oper: 0011 result: 0.4396178981810428
A_oper: 0111 result: 0.28698705300338284
A_oper: 1111 result: -0.012254705345268574
qc_result: 3.424287082602716


In [7]:
dev_numer = pl.device("default.qubit", wires=n_q, shots=n_shots)
@pl.qnode(dev_numer)
def qc_numerator(theta):
    ## ansatz unitary
    p = int((len(theta)-n_q)/(2*(n_q-1)))  # depth of ansatz
    for i in range(n_q):
        pl.RY(theta[i], wires=i)
    for iter in range(p):
        for i in range(n_q//2):
            pl.CZ(wires=[2*i, 2*i+1])
            pl.RY(theta[n_q+2*iter*(n_q-1)+2*i], wires=2*i)
            pl.RY(theta[n_q+2*iter*(n_q-1)+2*i+1], wires=2*i+1)
        if n_q != 2:
            for i in range((n_q-1)//2):
                pl.CZ(wires=[2*i+1, 2*i+2])
                pl.RY(theta[n_q+2*iter*(n_q-1)+2*(n_q//2)+2*i], wires=2*i+1)
                pl.RY(theta[n_q+2*iter*(n_q-1)+2*(n_q//2)+2*i+1], wires=2*i+2)
    # pl.Barrier(wires=[i for i in range(n_q)])
    ## measurement
    ### problem - RHS: constant function ###
    for i in range(n_q):
        pl.Hadamard(wires=i)
    return pl.probs(wires=[i for i in range(n_q)])

### estimate numerator ###
def cost_numerator(theta):
    results = qc_numerator(theta)
    numer_cost = prob_to_float(results, '0'*n_q)
    return numer_cost

In [8]:
### example
p = 2
num_param = n_q + n_q + 2*p*(n_q-1)
list_param = np.array([i for i in range(16)])*np.pi/8
test_theta = np.array([random.choice(list_param) for i in range(num_param)])
ansatz_state = np.array(get_ansatz_state(test_theta))
exact_result = np.dot(ansatz_state, normalized_rhs)
exact_result = exact_result**2
print('exact:', exact_result)
qc_result = cost_numerator(test_theta)
print('qc:', qc_result)

exact: (4.3545251446005494e-05+0j)
qc: 4.354525144600611e-05


In [9]:
def vqa_optimize(init_param, optimizer):
    eval_cost = 0
    def get_cost():
        def execute_circ(theta):
            nonlocal eval_cost
            eval_cost += 1
            global denom_cost, numer_cost
            denom_cost = cost_denominator(theta)
            numer_cost = cost_numerator(theta)
            cost = -(1/2)*numer_cost/denom_cost
            if eval_cost % 100 == 0:
                print(f"[step {eval_cost}] cost: {cost}")
            return cost
        return execute_circ
    cost_ftn = get_cost()
    if optimizer == 'BFGS':
        res = minimize(cost_ftn, init_param, method=optimizer)
    elif optimizer == 'L-BFGS-B':
        res = minimize(cost_ftn, init_param, method=optimizer)
    elif optimizer == 'COBYLA':
        res = minimize(cost_ftn, init_param, method=optimizer)
    return res, res.x

In [10]:
from scipy.optimize import minimize
p = 3
num_param = n_q + 2*p*(n_q-1)
list_param = np.array([i for i in range(32)])/8
param_list = np.array([random.choice(list_param) for i in range(num_param)])
init_param = param_list*np.pi
optimizer = 'L-BFGS-B'
result = vqa_optimize(init_param=init_param, optimizer=optimizer)
print(result[0])
qa_state = get_ansatz_state(result[1])
qa_state = np.array([qa_state[i].real for i in range(2**n_q)])
fidel_test = abs(np.dot(qa_state, normalized_sol))
print('fidel:', fidel_test)

[step 100] cost: -0.35093938973346167
[step 200] cost: -0.11253758671017257
[step 300] cost: -1.214247338647694
[step 400] cost: -0.003929893064250107
[step 500] cost: -2.314683196165292
[step 600] cost: -5.507264097607777
[step 700] cost: -7.490010307164189
[step 800] cost: -8.22275128278417
[step 900] cost: -9.038583854943246
[step 1000] cost: -10.676206499987957
[step 1100] cost: -10.426724753138794
[step 1200] cost: -11.902559637510953
[step 1300] cost: -12.17012549813355
[step 1400] cost: -12.416225509707475
[step 1500] cost: -12.513886643773894
[step 1600] cost: -12.547894162295641
[step 1700] cost: -12.630920877548672
[step 1800] cost: -12.667598361171036
[step 1900] cost: -12.70031305277886
[step 2000] cost: -12.71333663700697
[step 2100] cost: -12.72315764892493
[step 2200] cost: -12.73515421623837
[step 2300] cost: -12.740672900429747
[step 2400] cost: -12.748381666525416
[step 2500] cost: -12.748773668993328
[step 2600] cost: -12.749018431687647
[step 2700] cost: -12.7493453